# Ass1-Avon — 04 Modelling

**Stage:** Modelling (when justified) → Visualisation

**This notebook is optional, and skipping it is a legitimate result.** Plenty of good
analyses stop at description. Answer DECISION 8 before running anything below it.

> **This notebook is a stage template, not a mandatory step.** Use the notebooks your
> question needs and delete the rest — a descriptive project may never open
> `04_modeling.ipynb`, and a messy dataset may spend most of its life in `01`.
> See `CHECKLIST.md` for the full workflow.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Point this at your file in data/raw/. Everything below follows from it.
DATASET = 'REPLACE_ME.csv'

## DECISION 8 — Is a model justified at all?

A model earns its place by answering something description could not.

- **What the model would tell you that `02`/`03` did not:** _<...>_
- **Who uses the prediction, and for what:** _<a model nobody acts on is a costly chart>_
- **Is n large enough to fit and validate?** _<a curve through 8 points is a drawing>_
- **Decision:** _<model / do not model>_ because _<reason>_

If the answer is *do not model*, write that here and stop. Delete this notebook and go to
`reports/`. That is a finished analysis, not an incomplete one.

In [ ]:
from src.data.loading import load_raw
from src.data.cleaning import snake_case_columns, strip_whitespace

df = snake_case_columns(strip_whitespace(load_raw(DATASET)))
df.head()

## DECISION 9 — Predictors and target

- **Target:** _<column, and why it is the outcome that matters>_
- **Predictors:** _<columns, and why each is available *at prediction time*>_
- **Leakage check:** _<is any predictor a consequence of the target, or recorded after it?>_

A ranked correlation list from `02` is not an answer to this cell. Availability and
causal ordering decide it.

In [ ]:
# The baseline runs FIRST, always. Every later number is reported as a gain over this.
TARGET, PREDICTOR = None, None

if TARGET:
    from src.models.baseline import loocv_mean_baseline

    y = df[TARGET].dropna().to_numpy(float)
    print(f'mean-only baseline LOOCV RMSE: {loocv_mean_baseline(y):.4f}')
    print(f'(predicting {df[TARGET].mean():.4f} for every row)')
    print('\nA model that does not beat this is worse than no model.')
else:
    print('Set TARGET from DECISION 9.')

In [ ]:
from src.models.baseline import compare_models

if TARGET and PREDICTOR:
    res = compare_models(df, PREDICTOR, TARGET, degrees=(1, 2))
else:
    res = None
    print('Set TARGET and PREDICTOR from DECISION 9.')

## DECISION 10 — Which model, and why

`compare_models` reports and stops. It returns no winner because choosing one is the
judgement you are here to make.

- **Chosen:** _<model>_
- **Evidence:** _<LOOCV RMSE vs the baseline — not training R², which rises with
  flexibility whether or not the flexibility is real>_
- **Margin:** _<is the gap decisive at this n, or within noise?>_
- **Rejected because:** _<...>_

In [ ]:
from src.models.baseline import plot_comparison

if res:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    plot_comparison(df, res, ax)
    save_fig(fig, '04_model_comparison')
    plt.show()

In [ ]:
from src.analysis.hypothesis import permutation_test
from src.models.baseline import build_model, predict_with

# Is the gain over baseline more than chance? Assumption-light, which matters at small n.
if res:
    sub = df[[PREDICTOR, TARGET]].dropna()
    model, poly = build_model(sub[PREDICTOR], sub[TARGET], degree=1)
    fitted = predict_with(model, poly, sub[PREDICTOR])
    permutation_test(sub[TARGET].to_numpy(float), fitted, B=2000)

### Reading

_<Does the model beat the mean baseline out of sample? Is the permutation p small enough to
rule out chance? If either answer is no, say so plainly — a negative result honestly
reported is worth more than a rescued one.>_

## Imputation (only if DECISION 2 permitted it)

`impute` requires `method=` and returns a provenance record. Filled values must never be
indistinguishable from observed ones downstream.

In [ ]:
from src.models.imputation import METHODS, impute, write_provenance_artifacts

for name, description in METHODS.items():
    print(f'{name}:\n  {description}\n')

In [ ]:
# Set from DECISION 2. No default is supplied on purpose.
IMPUTE_COL, IMPUTE_METHOD = None, None

if IMPUTE_COL and IMPUTE_METHOD:
    df_imputed, provenance = impute(
        df, IMPUTE_COL, method=IMPUTE_METHOD,
        confidence='low',        # <- your assessment, stated
    )
    display(provenance)
    write_provenance_artifacts(df, df_imputed, provenance, stem='Ass1-Avon')
else:
    print('No imputation configured — correct if DECISION 2 said not to impute.')

## Limitations

Every model has these. Listing them is what separates a result from a claim.

- **Sample:** _<n, and what it does and does not represent>_
- **Validation:** _<how the out-of-sample estimate was produced, and its weakness>_
- **Assumptions:** _<linearity, independence, stable relationship over time>_
- **Not causal because:** _<no randomisation, confounders unmeasured, ...>_
- **Would change with:** _<more data, a control group, a different period>_

**Next stage:** the presentation-layer decision in `CHECKLIST.md`, then `reports/`.